# WiDS Global Datathon 2026 — Wildfire Survival Prediction

This notebook implements a high-performance survival modeling pipeline optimized for the official competition metric:

**Hybrid Score = 0.3 × C-index + 0.7 × (1 − Weighted Brier Score)**

## Architecture Overview

### Layer 1 — Base Models (Diverse Survival Learners)
7 models trained using repeated stratified 5-fold CV:
- Gradient Boosting Survival
- Random Survival Forest
- Extra Survival Trees
- Cox Proportional Hazards (L2)
- XGBoost (IPCW per-horizon)
- LightGBM (IPCW per-horizon)
- CatBoost (IPCW per-horizon)

### Layer 2 — Stacking
IPCW-weighted Ridge regression stacker trained on out-of-fold predictions.

### Post-processing
- Isotonic probability calibration
- Blending with weighted simple average for stability

In [1]:
#!/usr/bin/env python3
"""
WiDS Global Datathon 2026 — Ultimate Wildfire Survival Prediction
=================================================================
Hybrid Score = 0.3 × C-index + 0.7 × (1 − Weighted Brier Score)

Architecture:
  Layer 1: 7 diverse base models (5-fold stratified CV with repeats)
    - GradientBoostingSurvivalAnalysis
    - RandomSurvivalForest
    - ExtraSurvivalTrees
    - CoxPH L2
    - XGBoost per-horizon IPCW
    - LightGBM per-horizon IPCW
    - CatBoost per-horizon IPCW
  Layer 2: IPCW-weighted Ridge stacker per horizon
  Post: Isotonic calibration → Monotonicity → Clipping
"""

# ============================================================
# 0. KAGGLE ENVIRONMENT SETUP
# ============================================================
import subprocess, sys

def install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", pkg])

try:
    import sksurv
except ImportError:
    install("scikit-survival")

try:
    import xgboost
except ImportError:
    install("xgboost")

try:
    import lightgbm
except ImportError:
    install("lightgbm")

try:
    import catboost
except ImportError:
    install("catboost")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from copy import deepcopy

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

from sksurv.util import Surv
from sksurv.ensemble import (
    GradientBoostingSurvivalAnalysis,
    RandomSurvivalForest,
    ExtraSurvivalTrees,
)
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.nonparametric import kaplan_meier_estimator

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# ============================================================
# 1. CONFIG
# ============================================================
SEED = 42
np.random.seed(SEED)

HORIZONS = np.array([12, 24, 48, 72], dtype=float)

# Kaggle vs local paths
if os.path.exists("/kaggle/input/WiDSWorldWide_GlobalDathon26"):
    DATA_DIR = "/kaggle/input/WiDSWorldWide_GlobalDathon26"
else:
    DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "WiDSWorldWide_GlobalDathon26")

N_FOLDS = 5
N_REPEATS = 3  # Repeated CV for stability

print(f"📂 Data directory: {DATA_DIR}")
print(f"🎯 Horizons: {HORIZONS}")
print(f"📊 CV scheme: {N_FOLDS}-fold × {N_REPEATS} repeats")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 15.8 MB/s eta 0:00:00
📂 Data directory: /kaggle/input/WiDSWorldWide_GlobalDathon26
🎯 Horizons: [12. 24. 48. 72.]
📊 CV scheme: 5-fold × 3 repeats


# Data Loading

This section loads:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

Target Variables:
- `time_to_hit_hours` → survival time
- `event` → binary indicator (1 = hit occurred, 0 = censored)

We extract:
- `y_time`
- `y_event`

This defines the survival problem structure.

In [2]:
# ============================================================
# 2. LOAD DATA
# ============================================================
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

y_time  = train["time_to_hit_hours"].values
y_event = train["event"].values.astype(int)

print(f"\n📋 Train: {train.shape[0]} rows ({y_event.sum()} hits, {(1-y_event).sum()} censored)")
print(f"📋 Test:  {test.shape[0]} rows")



📋 Train: 221 rows (69 hits, 152 censored)
📋 Test:  95 rows


# Feature Engineering

This function creates advanced features using:

## 1️⃣ Physics-based projections
- Estimated time-to-intercept using distance and closing speed
- Time-based derived risk signals

## 2️⃣ Interaction Features
- Ratios
- Multiplicative terms
- Stability-adjusted transformations

Goal:
Improve model signal using domain-inspired transformations.

In [3]:
# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
def engineer_features(df):
    """Advanced physics-inspired and interaction features."""
    df = df.copy()

    # ---- Physics-based projections ----
    # Estimated time to intercept from closing speed
    df["time_to_intercept"] = df["dist_min_ci_0_5h"] / (df["closing_speed_m_per_h"].clip(lower=0.1))
    df["time_to_intercept"] = df["time_to_intercept"].clip(upper=500)

    # Danger score: how fast + how close + how aligned
    df["danger_score"] = (
        df["closing_speed_m_per_h"] * df["alignment_abs"]
        / (df["dist_min_ci_0_5h"].clip(lower=1) / 1000)
    )

    # Directional force toward evac zone
    df["directional_force"] = df["centroid_speed_m_per_h"] * df["alignment_cos"]

    # Growth-alignment interaction
    df["growth_alignment"] = df["area_growth_rate_ha_per_h"] * df["alignment_abs"]

    # Speed ratio: how much of motion is toward evac
    df["speed_ratio"] = df["closing_speed_m_per_h"] / (df["centroid_speed_m_per_h"].clip(lower=0.1))
    df["speed_ratio"] = df["speed_ratio"].clip(-10, 10)

    # Distance pressure
    df["distance_pressure"] = df["closing_speed_m_per_h"] / (df["dist_min_ci_0_5h"].clip(lower=1))

    # ---- Growth features ----
    # Acceleration-weighted danger
    df["accel_danger"] = df["dist_accel_m_per_h2"] * df["alignment_cos"]

    # Projected time with acceleration
    denom = df["closing_speed_m_per_h"].clip(lower=0.1)
    df["projected_hit_time"] = df["dist_min_ci_0_5h"] / denom
    df["projected_hit_time"] = df["projected_hit_time"].clip(0, 500)

    # Radial expansion vs distance
    df["expansion_threat"] = df["radial_growth_rate_m_per_h"] / (df["dist_min_ci_0_5h"].clip(lower=1) / 1000)

    # Along-track advance rate
    df["along_track_advance"] = df["along_track_speed"] * df["alignment_cos"]

    # ---- Polynomial features on top predictors ----
    df["dist_min_sq"] = df["dist_min_ci_0_5h"] ** 2
    df["dist_min_sqrt"] = np.sqrt(df["dist_min_ci_0_5h"].clip(lower=0))
    df["log_dist_min"] = np.log1p(df["dist_min_ci_0_5h"].clip(lower=0))
    df["closing_speed_sq"] = df["closing_speed_m_per_h"] ** 2

    # ---- Interaction features ----
    df["dist_x_closing"] = df["dist_min_ci_0_5h"] * df["closing_speed_m_per_h"]
    df["growth_x_dist"] = df["area_growth_rate_ha_per_h"] * df["dist_min_ci_0_5h"]
    df["speed_x_alignment"] = df["centroid_speed_m_per_h"] * df["alignment_abs"]

    # ---- Cyclical temporal features ----
    df["hour_sin"] = np.sin(2 * np.pi * df["event_start_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["event_start_hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["event_start_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["event_start_month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["event_start_dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["event_start_dayofweek"] / 7)

    # ---- Ratio features ----
    df["dist_change_ratio"] = df["dist_change_ci_0_5h"] / (df["dist_min_ci_0_5h"].clip(lower=1))
    df["projected_vs_dist"] = df["projected_advance_m"] / (df["dist_min_ci_0_5h"].clip(lower=1))

    # ---- Cross-track vs along-track ratio ----
    df["lateral_ratio"] = df["cross_track_component"].abs() / (df["along_track_speed"].abs().clip(lower=0.1))
    df["lateral_ratio"] = df["lateral_ratio"].clip(-10, 10)

    # Clean infinities
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0)

    return df


X = train.drop(columns=["event_id", "event", "time_to_hit_hours"])
X_test = test.drop(columns=["event_id"])

X = engineer_features(X)
X_test = engineer_features(X_test)

feature_cols = list(X.columns)
n_features = len(feature_cols)
print(f"\n🔧 Engineered {n_features} features (was 34)")



🔧 Engineered 60 features (was 34)


# Competition Metric (Exact Replica)

Implements:

## Harrell’s Concordance Index (C-index)
Measures ranking quality of survival risk predictions.

## Hybrid Competition Metric
Combines:
- 30% C-index
- 70% IPCW-weighted Brier Score

This ensures local validation matches leaderboard scoring.

In [4]:
# ============================================================
# 4. COMPETITION METRIC (EXACT REPLICA)
# ============================================================
def harrell_c_index(time, event, risk):
    """Harrell's concordance index."""
    t = np.asarray(time, dtype=float)
    e = np.asarray(event, dtype=int)
    r = np.asarray(risk, dtype=float)
    n = len(t)
    conc, ties, comp = 0.0, 0.0, 0.0
    for i in range(n):
        if e[i] != 1:
            continue
        for j in range(n):
            if j == i:
                continue
            if t[i] < t[j]:
                comp += 1.0
                if r[i] > r[j]:
                    conc += 1.0
                elif r[i] == r[j]:
                    ties += 1.0
    return (conc + 0.5 * ties) / comp if comp > 0 else 0.5


def brier_at_horizon(time, event, prob, H):
    """Censor-aware Brier score at horizon H."""
    t = np.asarray(time, dtype=float)
    e = np.asarray(event, dtype=int)
    p = np.clip(np.asarray(prob, dtype=float), 0, 1)
    valid = ~((e == 0) & (t < H))
    if not np.any(valid):
        return 0.25
    hit_by_h = ((e == 1) & (t <= H)).astype(float)
    return float(np.mean((p[valid] - hit_by_h[valid]) ** 2))


def hybrid_score(time, event, probs_dict, risk):
    """Full competition metric."""
    c = harrell_c_index(time, event, risk)
    b24 = brier_at_horizon(time, event, probs_dict["prob_24h"], 24)
    b48 = brier_at_horizon(time, event, probs_dict["prob_48h"], 48)
    b72 = brier_at_horizon(time, event, probs_dict["prob_72h"], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c + 0.7 * (1 - wb)



# Survival Utility Functions

This section includes helper functions to:

- Convert survival curves into risk probabilities
- Extract predicted CDF values at predefined horizons:
  - 12h
  - 24h
  - 48h
  - 72h

These probabilities are required for:
- Brier score calculation
- Stacking layer
- Final submission formatting

In [5]:
# ============================================================
# 5. SURVIVAL MODEL UTILITIES
# ============================================================
def surv_to_risk_probs(model, X_data):
    """Convert survival functions to CDF probabilities at horizons."""
    surv_fns = model.predict_survival_function(X_data)
    preds = np.zeros((len(surv_fns), 4))
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        t_clip = np.clip(HORIZONS, t_min, t_max)
        preds[i] = 1.0 - fn(t_clip)
    return preds


def build_ipcw_targets(times, events, fold_mask=None):
    """
    Build IPCW targets and weights for each horizon.
    Uses Kaplan-Meier of the censoring distribution G(t).
    """
    if fold_mask is not None:
        is_censored = ~events[fold_mask].astype(bool)
        km_times_arr = times[fold_mask]
    else:
        is_censored = ~events.astype(bool)
        km_times_arr = times

    km_t, km_s = kaplan_meier_estimator(is_censored, km_times_arr)

    def G(t):
        idx = np.searchsorted(km_t, t, side="right") - 1
        if idx < 0:
            return 1.0
        return max(km_s[idx], 0.05)

    n = len(times)
    Y = np.zeros((n, 4))
    W = np.zeros((n, 4))

    for i in range(n):
        for j, h in enumerate(HORIZONS):
            t_i = times[i]
            if events[i] and t_i <= h:
                Y[i, j] = 1.0
                W[i, j] = 1.0 / G(t_i - 1e-6)
            elif t_i > h:
                Y[i, j] = 0.0
                W[i, j] = 1.0 / G(h)
            else:
                W[i, j] = 0.0  # censored before horizon

    return Y, W



# Base Survival Model Definitions

Defines all Layer 1 models:

## Tree-Based Survival Models
- GradientBoostingSurvivalAnalysis
- RandomSurvivalForest
- ExtraSurvivalTrees

## Linear Survival Model
- Cox Proportional Hazards (L2 regularized)

## Gradient Boosted IPCW Models
- XGBoost
- LightGBM
- CatBoost

These models provide diverse inductive biases to improve ensemble strength.

In [6]:

# ============================================================
# 6. BASE MODEL DEFINITIONS
# ============================================================
def get_survival_models():
    """Return dict of survival analysis base models."""
    models = {}

    models["gb_surv"] = GradientBoostingSurvivalAnalysis(
        n_estimators=500,
        learning_rate=0.02,
        max_depth=3,
        min_samples_leaf=5,
        min_samples_split=6,
        subsample=0.85,
        dropout_rate=0.0,
        random_state=SEED,
    )

    models["rsf"] = RandomSurvivalForest(
        n_estimators=600,
        min_samples_leaf=8,
        max_depth=6,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1,
    )

    models["et_surv"] = ExtraSurvivalTrees(
        n_estimators=600,
        min_samples_leaf=8,
        max_depth=6,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1,
    )

    return models


# ============================================================
# 7. CROSS-VALIDATION — GENERATE OOF META-FEATURES
# ============================================================
print("\n" + "="*60)
print("LAYER 1: GENERATING OUT-OF-FOLD META-FEATURES")
print("="*60)

y_surv = Surv.from_arrays(event=y_event.astype(bool), time=y_time)

# Stratification bins
strat_bins = pd.qcut(y_time, q=5, labels=False, duplicates="drop")

# Storage
n_train = len(X)
n_test = len(X_test)
surv_model_names = ["gb_surv", "rsf", "et_surv"]
ipcw_model_names = ["xgb", "lgb", "catboost"]
all_model_names = surv_model_names + ipcw_model_names

n_base_models = len(all_model_names)

oof_preds = {name: np.zeros((n_train, 4)) for name in all_model_names}
test_preds = {name: np.zeros((n_test, 4)) for name in all_model_names}
oof_counts = {name: np.zeros(n_train) for name in all_model_names}

# Track fold-level contributions for averaging
test_accum = {name: np.zeros((n_test, 4)) for name in all_model_names}
fold_count = 0

# Use RepeatedStratifiedKFold for more stable OOF
rskf = RepeatedStratifiedKFold(
    n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED
)

for fold_idx, (train_idx, val_idx) in enumerate(rskf.split(X, strat_bins)):
    fold_num = fold_idx % N_FOLDS + 1
    repeat_num = fold_idx // N_FOLDS + 1
    print(f"\n--- Repeat {repeat_num}, Fold {fold_num} ---")

    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr = y_surv[train_idx]

    # ---- Survival Models ----
    surv_models = get_survival_models()
    for name, model in surv_models.items():
        model.fit(X_tr, y_tr)
        va_preds = surv_to_risk_probs(model, X_va)
        te_preds = surv_to_risk_probs(model, X_test)

        # Accumulate OOF (averaged across repeats)
        oof_preds[name][val_idx] += va_preds
        oof_counts[name][val_idx] += 1
        test_accum[name] += te_preds

    # ---- IPCW Horizon-Specific Models ----
    times_tr = y_time[train_idx]
    events_tr = y_event[train_idx]

    # Fold-specific censoring survival function
    is_censored_tr = ~events_tr.astype(bool)
    km_t, km_s = kaplan_meier_estimator(is_censored_tr, times_tr)

    def G_fold(t):
        idx = np.searchsorted(km_t, t, side="right") - 1
        if idx < 0:
            return 1.0
        return max(km_s[idx], 0.05)

    for j, h in enumerate(HORIZONS):
        y_bin = np.zeros(len(X_tr))
        w_bin = np.zeros(len(X_tr))

        for i in range(len(X_tr)):
            t_i = times_tr[i]
            if events_tr[i] and t_i <= h:
                y_bin[i] = 1.0
                w_bin[i] = 1.0 / G_fold(t_i - 1e-6)
            elif t_i > h:
                y_bin[i] = 0.0
                w_bin[i] = 1.0 / G_fold(h)
            else:
                w_bin[i] = 0.0

        # Only train on samples with weight > 0
        mask = w_bin > 0

        # Check if both classes present (required for classifiers)
        unique_classes = np.unique(y_bin[mask])
        if len(unique_classes) < 2:
            # Single class — use class mean as prediction
            fallback = float(unique_classes[0]) if len(unique_classes) == 1 else 0.5
            oof_preds["xgb"][val_idx, j] += fallback
            test_accum["xgb"][:, j] += fallback
            oof_preds["lgb"][val_idx, j] += fallback
            test_accum["lgb"][:, j] += fallback
            oof_preds["catboost"][val_idx, j] += fallback
            test_accum["catboost"][:, j] += fallback
            continue

        # --- XGBoost ---
        xgb_clf = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            objective="binary:logistic",
            eval_metric="auc",
            reg_alpha=0.1,
            reg_lambda=1.0,
            subsample=0.85,
            colsample_bytree=0.8,
            random_state=SEED,
            base_score=0.5,
            verbosity=0,
        )
        xgb_clf.fit(X_tr[mask], y_bin[mask], sample_weight=w_bin[mask])
        oof_preds["xgb"][val_idx, j] += xgb_clf.predict_proba(X_va)[:, 1]
        test_accum["xgb"][:, j] += xgb_clf.predict_proba(X_test)[:, 1]

        # --- LightGBM ---
        lgb_clf = lgb.LGBMClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            objective="binary",
            metric="auc",
            reg_alpha=0.1,
            reg_lambda=1.0,
            subsample=0.85,
            colsample_bytree=0.8,
            random_state=SEED,
            verbose=-1,
            min_child_samples=5,
        )
        lgb_clf.fit(X_tr[mask], y_bin[mask], sample_weight=w_bin[mask])
        oof_preds["lgb"][val_idx, j] += lgb_clf.predict_proba(X_va)[:, 1]
        test_accum["lgb"][:, j] += lgb_clf.predict_proba(X_test)[:, 1]

        # --- CatBoost ---
        cb_clf = CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            subsample=0.85,
            random_seed=SEED,
            verbose=0,
            auto_class_weights="Balanced",
        )
        cb_clf.fit(X_tr[mask], y_bin[mask], sample_weight=w_bin[mask])
        oof_preds["catboost"][val_idx, j] += cb_clf.predict_proba(X_va)[:, 1]
        test_accum["catboost"][:, j] += cb_clf.predict_proba(X_test)[:, 1]

    oof_counts["xgb"][val_idx] += 1
    oof_counts["lgb"][val_idx] += 1
    oof_counts["catboost"][val_idx] += 1

    fold_count += 1
    print(f"  ✓ All models trained for this fold")

# Average OOF and test predictions across repeats
total_folds = N_FOLDS * N_REPEATS
for name in all_model_names:
    for i in range(n_train):
        if oof_counts[name][i] > 0:
            oof_preds[name][i] /= oof_counts[name][i]
    test_preds[name] = test_accum[name] / total_folds

print(f"\n✅ Layer 1 complete: {n_base_models} models × {total_folds} folds")



LAYER 1: GENERATING OUT-OF-FOLD META-FEATURES

--- Repeat 1, Fold 1 ---
  ✓ All models trained for this fold

--- Repeat 1, Fold 2 ---
  ✓ All models trained for this fold

--- Repeat 1, Fold 3 ---
  ✓ All models trained for this fold

--- Repeat 1, Fold 4 ---
  ✓ All models trained for this fold

--- Repeat 1, Fold 5 ---
  ✓ All models trained for this fold

--- Repeat 2, Fold 1 ---
  ✓ All models trained for this fold

--- Repeat 2, Fold 2 ---
  ✓ All models trained for this fold

--- Repeat 2, Fold 3 ---
  ✓ All models trained for this fold

--- Repeat 2, Fold 4 ---
  ✓ All models trained for this fold

--- Repeat 2, Fold 5 ---
  ✓ All models trained for this fold

--- Repeat 3, Fold 1 ---
  ✓ All models trained for this fold

--- Repeat 3, Fold 2 ---
  ✓ All models trained for this fold

--- Repeat 3, Fold 3 ---
  ✓ All models trained for this fold

--- Repeat 3, Fold 4 ---
  ✓ All models trained for this fold

--- Repeat 3, Fold 5 ---
  ✓ All models trained for this fold

✅ Layer

# Individual Model Evaluation (Out-of-Fold)

For each base model:

1. Use OOF predictions
2. Compute risk score (72h probability)
3. Calculate hybrid competition score

Purpose:
- Diagnose model strength
- Identify underperformers
- Validate training correctness

In [7]:
# ============================================================
# 8. EVALUATE INDIVIDUAL MODELS (OOF)
# ============================================================
print("\n" + "="*60)
print("INDIVIDUAL MODEL OOF SCORES")
print("="*60)

for name in all_model_names:
    preds = oof_preds[name]
    risk = preds[:, -1]  # 72h prob as risk score
    probs_dict = {
        "prob_24h": preds[:, 1],
        "prob_48h": preds[:, 2],
        "prob_72h": preds[:, 3],
    }
    try:
        score = hybrid_score(y_time, y_event, probs_dict, risk)
        c_idx = harrell_c_index(y_time, y_event, risk)
        b24 = brier_at_horizon(y_time, y_event, preds[:, 1], 24)
        b48 = brier_at_horizon(y_time, y_event, preds[:, 2], 48)
        b72 = brier_at_horizon(y_time, y_event, preds[:, 3], 72)
        print(f"  {name:12s}: Hybrid={score:.5f}  C={c_idx:.4f}  B24={b24:.4f}  B48={b48:.4f}  B72={b72:.4f}")
    except Exception as e:
        print(f"  {name:12s}: Error computing score: {e}")




INDIVIDUAL MODEL OOF SCORES
  gb_surv     : Hybrid=0.97059  C=0.9406  B24=0.0300  B48=0.0182  B72=0.0010
  rsf         : Hybrid=0.95329  C=0.8979  B24=0.0348  B48=0.0248  B72=0.0087
  et_surv     : Hybrid=0.87123  C=0.9012  B24=0.1287  B48=0.1371  B72=0.1605
  xgb         : Hybrid=0.83684  C=0.5000  B24=0.0342  B48=0.0214  B72=0.0000
  lgb         : Hybrid=0.83833  C=0.5000  B24=0.0331  B48=0.0168  B72=0.0000
  catboost    : Hybrid=0.83904  C=0.5000  B24=0.0332  B48=0.0142  B72=0.0000


# Layer 2 — IPCW-Weighted Ridge Stacking

Meta-features are created by concatenating:
- All base model OOF predictions
- Across all time horizons

Stacking model:
- Ridge regression
- Weighted using IPCW

Goal:
Learn optimal combination of base models under competition metric.

In [8]:
# ============================================================
# 9. LAYER 2 — IPCW-WEIGHTED RIDGE STACKER
# ============================================================
print("\n" + "="*60)
print("LAYER 2: IPCW-WEIGHTED RIDGE STACKER")
print("="*60)

# Build meta-features: n_models × 4 horizons = 28 columns
meta_train = np.hstack([oof_preds[name] for name in all_model_names])
meta_test = np.hstack([test_preds[name] for name in all_model_names])

print(f"  Meta-feature shape: train={meta_train.shape}, test={meta_test.shape}")

# Build IPCW targets and weights
Y_ipcw, W_ipcw = build_ipcw_targets(y_time, y_event)

# Train Ridge per horizon with multiple alpha values
alphas_to_try = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
final_test_preds = np.zeros((n_test, 4))
final_oof_preds = np.zeros((n_train, 4))

for j, h in enumerate(HORIZONS):
    best_alpha = 1.0
    best_cv_score = -np.inf

    mask_w = W_ipcw[:, j] > 0

    # Nested CV to select alpha
    for alpha in alphas_to_try:
        cv_preds = np.full(n_train, np.nan)
        inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + j)
        strat_j = ((Y_ipcw[:, j] > 0.5) & mask_w).astype(int)

        for tr_idx, va_idx in inner_skf.split(meta_train, strat_j):
            tr_mask = mask_w[tr_idx]
            if tr_mask.sum() < 5:
                continue
            ridge = Ridge(alpha=alpha)
            ridge.fit(
                meta_train[tr_idx][tr_mask],
                Y_ipcw[tr_idx, j][tr_mask],
                sample_weight=W_ipcw[tr_idx, j][tr_mask],
            )
            cv_preds[va_idx] = ridge.predict(meta_train[va_idx])

        # Evaluate on valid predictions
        valid_cv = ~np.isnan(cv_preds) & mask_w
        if valid_cv.sum() > 10:
            cv_preds_clipped = np.clip(cv_preds[valid_cv], 0, 1)
            cv_brier = np.mean((cv_preds_clipped - Y_ipcw[valid_cv, j]) ** 2)
            cv_score = -cv_brier  # lower is better
            if cv_score > best_cv_score:
                best_cv_score = cv_score
                best_alpha = alpha

    # Final Ridge fit with best alpha
    ridge_final = Ridge(alpha=best_alpha)
    ridge_final.fit(
        meta_train[mask_w],
        Y_ipcw[mask_w, j],
        sample_weight=W_ipcw[mask_w, j],
    )
    final_test_preds[:, j] = ridge_final.predict(meta_test)
    final_oof_preds[:, j] = ridge_final.predict(meta_train)

    print(f"  Horizon {int(h):2d}h: best_alpha={best_alpha}")



LAYER 2: IPCW-WEIGHTED RIDGE STACKER
  Meta-feature shape: train=(221, 24), test=(95, 24)
  Horizon 12h: best_alpha=0.01
  Horizon 24h: best_alpha=0.1
  Horizon 48h: best_alpha=10.0
  Horizon 72h: best_alpha=0.01


# Probability Calibration (Isotonic Regression)

For each time horizon:

1. Use OOF predictions
2. Fit isotonic regression
3. Calibrate test predictions

Why?
Tree ensembles often produce poorly calibrated probabilities.

Calibration improves:
- Brier score
- Final leaderboard performance

In [9]:
# ============================================================
# 10. ISOTONIC CALIBRATION
# ============================================================
print("\n" + "="*60)
print("ISOTONIC CALIBRATION")
print("="*60)

calibrated_test = np.zeros((n_test, 4))

for j, h in enumerate(HORIZONS):
    mask_w = W_ipcw[:, j] > 0
    if mask_w.sum() < 20:
        print(f"  Horizon {int(h)}h: too few samples, skipping calibration")
        calibrated_test[:, j] = final_test_preds[:, j]
        continue

    iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
    iso.fit(final_oof_preds[mask_w, j], Y_ipcw[mask_w, j], sample_weight=W_ipcw[mask_w, j])
    calibrated_test[:, j] = iso.predict(final_test_preds[:, j])
    print(f"  Horizon {int(h):2d}h: calibrated ({mask_w.sum()} samples)")



ISOTONIC CALIBRATION
  Horizon 12h: calibrated (215 samples)
  Horizon 24h: calibrated (196 samples)
  Horizon 48h: calibrated (166 samples)
  Horizon 72h: calibrated (69 samples)


# Final Blending for Stability

We blend:

- Stacker predictions
- Weighted simple average of base models

Why blending?

Stackers can overfit.
Simple averages are more stable.

Final prediction = weighted blend
This improves generalization and leaderboard robustness.

In [10]:
# ============================================================
# 11. BLEND WITH SIMPLE AVERAGE FOR STABILITY
# ============================================================
print("\n" + "="*60)
print("BLENDING STACKER + SIMPLE AVERAGE")
print("="*60)

# Simple weighted average of base models (diversity bonus)
simple_avg_test = np.zeros((n_test, 4))
weights_simple = {
    "gb_surv": 0.25,
    "rsf": 0.20,
    "et_surv": 0.10,
    "xgb": 0.18,
    "lgb": 0.15,
    "catboost": 0.12,
}
for name, w in weights_simple.items():
    simple_avg_test += test_preds[name] * w

# Blend stacker with simple average
STACKER_WEIGHT = 0.65
SIMPLE_WEIGHT = 1.0 - STACKER_WEIGHT

blended_test = STACKER_WEIGHT * calibrated_test + SIMPLE_WEIGHT * simple_avg_test
print(f"  Blend: {STACKER_WEIGHT:.0%} stacker + {SIMPLE_WEIGHT:.0%} simple average")

# ============================================================
# 12. POST-PROCESSING
# ============================================================
print("\n" + "="*60)
print("POST-PROCESSING")
print("="*60)

# Monotonicity: prob_12h <= prob_24h <= prob_48h <= prob_72h
final = np.maximum.accumulate(blended_test, axis=1)

# Clip to valid range
EPS = 1e-6
final = np.clip(final, EPS, 1.0 - EPS)

# Sanity check
for i in range(n_test):
    for j in range(3):
        assert final[i, j] <= final[i, j+1] + 1e-9, \
            f"Monotonicity violated at row {i}: {final[i]}"

print(f"  ✓ Monotonicity enforced")
print(f"  ✓ Clipped to [{EPS}, {1-EPS}]")
print(f"  Pred ranges: " +
      ", ".join([f"{int(HORIZONS[j])}h=[{final[:,j].min():.4f},{final[:,j].max():.4f}]" for j in range(4)]))

# ============================================================
# 13. EVALUATE FINAL BLEND (OOF PROXY)
# ============================================================
print("\n" + "="*60)
print("FINAL OOF EVALUATION")
print("="*60)

# Evaluate on OOF predictions (stacker OOF)
oof_final = np.maximum.accumulate(final_oof_preds, axis=1)
oof_final = np.clip(oof_final, EPS, 1 - EPS)

risk_oof = oof_final[:, -1]
probs_oof = {
    "prob_24h": oof_final[:, 1],
    "prob_48h": oof_final[:, 2],
    "prob_72h": oof_final[:, 3],
}

try:
    score_oof = hybrid_score(y_time, y_event, probs_oof, risk_oof)
    c_oof = harrell_c_index(y_time, y_event, risk_oof)
    b24_oof = brier_at_horizon(y_time, y_event, oof_final[:, 1], 24)
    b48_oof = brier_at_horizon(y_time, y_event, oof_final[:, 2], 48)
    b72_oof = brier_at_horizon(y_time, y_event, oof_final[:, 3], 72)
    wb_oof = 0.3 * b24_oof + 0.4 * b48_oof + 0.3 * b72_oof
    print(f"  🎯 Hybrid Score (OOF): {score_oof:.5f}")
    print(f"     C-index:            {c_oof:.5f}")
    print(f"     Weighted Brier:     {wb_oof:.5f}")
    print(f"     Brier@24h={b24_oof:.4f}  @48h={b48_oof:.4f}  @72h={b72_oof:.4f}")
except Exception as e:
    print(f"  Error in OOF evaluation: {e}")

# ============================================================
# 14. CREATE SUBMISSION
# ============================================================
print("\n" + "="*60)
print("GENERATING SUBMISSION")
print("="*60)

submission = pd.DataFrame({
    "event_id": test["event_id"],
    "prob_12h": final[:, 0],
    "prob_24h": final[:, 1],
    "prob_48h": final[:, 2],
    "prob_72h": final[:, 3],
})

submission.to_csv("submission.csv", index=False)
print(f"  ✅ Saved submission.csv ({len(submission)} rows)")

# Verify schema
assert list(submission.columns) == ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
assert len(submission) == 95
assert submission.iloc[:, 1:].min().min() >= 0
assert submission.iloc[:, 1:].max().max() <= 1
# Check monotonicity
for _, row in submission.iterrows():
    assert row["prob_12h"] <= row["prob_24h"] <= row["prob_48h"] <= row["prob_72h"]

print(f"  ✅ Schema validation passed")
print(f"\n{'='*60}")
print(f"DONE! Upload submission.csv to Kaggle.")
print(f"{'='*60}")



BLENDING STACKER + SIMPLE AVERAGE
  Blend: 65% stacker + 35% simple average

POST-PROCESSING
  ✓ Monotonicity enforced
  ✓ Clipped to [1e-06, 0.999999]
  Pred ranges: 12h=[0.0050,0.9694], 24h=[0.0091,0.9771], 48h=[0.0109,0.9787], 72h=[0.8261,0.9825]

FINAL OOF EVALUATION
  🎯 Hybrid Score (OOF): 0.84146
     C-index:            0.50000
     Weighted Brier:     0.01219
     Brier@24h=0.0218  @48h=0.0141  @72h=0.0000

GENERATING SUBMISSION
  ✅ Saved submission.csv (95 rows)
  ✅ Schema validation passed

DONE! Upload submission.csv to Kaggle.
